In [ ]:
!ls

In [ ]:
%cd /content

In [ ]:
!git clone https://github.com/nabin2004/AOS.git /content/AOS
%cd /content/AOS

In [ ]:
!export UV_LINK_MODE=copy

In [ ]:
# HF_TOKEN is loaded from Colab secrets in the next cell
pass


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")
except Exception:
    pass
os.environ["WANDB_PROJECT_SFT"] = "aos-sft"
os.environ["WANDB_RUN_NAME"] = "gemma4-manim-sft"


In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])


In [ ]:
!uv sync --package sft

In [ ]:
!uv run --package sft python apps/sft/run.py --colab --epochs 5

In [ ]:
!uv run --package sft python apps/sft/upload_adapter.py --adapter-dir /content/gemma4-31b-manim-ft --colab

In [ ]:
# cd apps/sft
# !uv run python merge_adapter.py \
#   --adapter-dir ./gemma4-manim-ft \
#   --output-dir ./gemma4-manim-merged

In [ ]:
!uv run python merge_adapter.py \
  --adapter-dir ./gemma4-31b-manim-ft \
  --output-dir ./gemma4-31b-manim-merged \
  --push-to-hub

## GGUF format

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build -j

In [ ]:
!pwd

In [ ]:
!sudo apt-get install zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# %cd apps/sft

# 1. Merge (if not done already)
!uv run python merge_adapter.py \
  --adapter-dir /content/gemma4-31b-manim-ft \
  --output-dir ./gemma4-31b-manim-merged

# # 2. Export GGUF + create Ollama model
# !export LLAMA_CPP_DIR=~/llama.cpp
# !uv run python export_gguf.py \
#   --model-dir ./gemma4-manim-merged \
#   --output-dir ./gemma4-manim-gguf

# 2b. Export GGUF + push to Hugging Face (separate repo from merged HF weights)
!uv run python export_gguf.py \
  --model-dir ./gemma4-31b-manim-merged \
  --output-dir ./gemma4-31b-manim-gguf \
  --push-to-hub \
  --skip-ollama-create


# # 3. Run locally
# ollama run aos-gemma4-manim


In [ ]:
print("DONE!")

In [ ]:
!uv run --package sft python apps/sft/infer.py \
  --adapter-dir /content/gemma4-manim-ft --colab \
  --prompt "Create a short Manim scene explaining eigenvectors in 2D."

In [ ]:
!uv add dbos

In [ ]:
!cd apps/sft
!uv run python apps/sft/infer.py --adapter-dir /content/gemma4-manim-ft \
  --prompt "Animate a unit circle morphing into an ellipse under a 2x2 matrix."

In [ ]:
!uv run --package sft python apps/sft/infer.py \
  --adapter-dir /content/gemma4-manim-ft \
  --colab \
  --prompt "Create a short Manim scene explaining eigenvectors in 2D."

In [ ]:
import shutil
import os
from google.colab import files

# Path to your directory
dir_path = "/content/AOS/apps/agents/workspace"

# Create a zip archive
archive_name = "workspace_backup"
shutil.make_archive(archive_name, 'zip', dir_path)

# Download the zip file
files.download(f"{archive_name}.zip")

# Optionally remove the zip after download (cleanup)
# os.remove(f"{archive_name}.zip")